SHIPMENTS.JSON DATA PARSING FROM JSON TO DATAFRAME AND DATAFRAME TO MYSQL.

In [1]:
import json

In [2]:
with open("shipments.json", "r") as shipments:
    shipments_json = shipments.read()

In [3]:
shipments_py = json.loads(shipments_json)
shipments_py

[{'shipment_id': 'dc84cc15',
  'order_date': '2025-06-12',
  'origin': 'North Dawnfort',
  'destination': 'Ginafort',
  'weight': 43.52,
  'courier_id': 'b554e6a4',
  'status': 'Cancelled',
  'delivery_date': None},
 {'shipment_id': 'ca536e73',
  'order_date': '2025-06-27',
  'origin': 'North Williamchester',
  'destination': 'Christinechester',
  'weight': 19.31,
  'courier_id': '9b44d9ca',
  'status': 'Cancelled',
  'delivery_date': None},
 {'shipment_id': '704fa19e',
  'order_date': '2025-09-09',
  'origin': 'New Colleen',
  'destination': 'West Jared',
  'weight': 27.97,
  'courier_id': 'f8c3e908',
  'status': 'Delivered',
  'delivery_date': '2025-09-13'},
 {'shipment_id': 'aad8a598',
  'order_date': '2025-07-25',
  'origin': 'South Ashleystad',
  'destination': 'Jonestown',
  'weight': 18.74,
  'courier_id': '0de3fc1b',
  'status': 'Cancelled',
  'delivery_date': None},
 {'shipment_id': 'e120d4ba',
  'order_date': '2026-01-06',
  'origin': 'Darinton',
  'destination': 'West Jared'

In [4]:
import pandas as pd

In [5]:
df4 = pd.DataFrame(shipments_py)
df4

,shipment_id,order_date,origin,destination,weight,courier_id,status,delivery_date
0,dc84cc15,2025-06-12,North Dawnfort,Ginafort,43.52,b554e6a4,Cancelled,None
1,ca536e73,2025-06-27,North Williamchester,Christinechester,19.31,9b44d9ca,Cancelled,None
2,704fa19e,2025-09-09,New Colleen,West Jared,27.97,f8c3e908,Delivered,2025-09-13
3,aad8a598,2025-07-25,South Ashleystad,Jonestown,18.74,0de3fc1b,Cancelled,None
4,e120d4ba,2026-01-06,Darinton,West Jared,39.30,44def2f4,In Transit,None
...,...,...,...,...,...,...,...,...
69995,33f9c844,2025-08-09,West Jared,South Brianside,6.66,09470953,Cancelled,None
69996,6ca403d1,2025-09-05,Kimfort,East Johnshire,16.78,353f1b39,Cancelled,None
69997,59a42fb4,2025-05-29,Hollyshire,Port Sandrachester,6.97,acb241ba,Cancelled,None
69998,3cfb7692,2026-02-18,South Ashleystad,New Colleen,49.57,409aec25,Pending,None


In [6]:
shipments_by_origins = df4.groupby("origin")
print(shipments_by_origins)

In [8]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   shipment_id    70000 non-null  object 
 1   order_date     70000 non-null  object 
 2   origin         70000 non-null  object 
 3   destination    70000 non-null  object 
 4   weight         70000 non-null  float64
 5   courier_id     70000 non-null  object 
 6   status         70000 non-null  object 
 7   delivery_date  17336 non-null  object 
dtypes: float64(1), object(7)
memory usage: 4.3+ MB


In [9]:
df4.rename(columns={"status": "delivery_status"}, inplace = True)

In [11]:
df4["shipment_id"] = df4["shipment_id"].str.strip()

In [12]:
df4["origin"] = df4["origin"].str.strip()

In [13]:
df4["destination"] = df4["destination"].str.strip()

In [14]:
df4["courier_id"] = df4["courier_id"].str.strip()

In [15]:
df4["delivery_status"] = df4["delivery_status"].str.strip()

In [16]:
df4["order_date"] = pd.to_datetime(df4["order_date"]).dt.date

In [17]:
df4["delivery_date"] = pd.to_datetime(df4["delivery_date"]).dt.date

In [18]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   shipment_id      70000 non-null  object 
 1   order_date       70000 non-null  object 
 2   origin           70000 non-null  object 
 3   destination      70000 non-null  object 
 4   weight           70000 non-null  float64
 5   courier_id       70000 non-null  object 
 6   delivery_status  70000 non-null  object 
 7   delivery_date    17336 non-null  object 
dtypes: float64(1), object(7)
memory usage: 4.3+ MB


In [8]:
df4["courier_id"].nunique()

1000

In [20]:
df4.drop_duplicates(subset=['shipment_id'], keep='first', inplace=True)

In [22]:
df4["shipment_id"].duplicated().value_counts()

shipment_id
False    69999
Name: count, dtype: int64

In [23]:
df4.duplicated().value_counts()

False    69999
Name: count, dtype: int64

In [24]:
df4["courier_id"].duplicated().sum()

np.int64(68999)

In [1]:
import mysql.connector

In [82]:
conn = mysql.connector.connect(host = "localhost", user = "root", password = "9095", database= "smart_logistics")
cursor = conn.cursor()

In [83]:
for index, row in df4.iterrows():
    sql = "insert into shipments (shipment_id, order_date, origin, destination,	weight,	courier_id, delivery_status, delivery_date) values (%s, %s, %s, %s, %s, %s, %s, %s)"
    cursor.execute(sql, (row["shipment_id"], row["order_date"], row["origin"], row["destination"], row["weight"], row["courier_id"], row["delivery_status"], row["delivery_date"]))
conn.commit()
cursor.close()
conn.close()